# Retrieval-Augmented Generation (RAG) — Implementation / 구현

**Paper**: Lewis, P. et al. "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks." *NeurIPS 33*, 2020. arXiv:2005.11401.

This notebook implements a tiny end-to-end RAG pipeline: a small fact corpus, a dense retriever (cosine similarity over embedded passages), a template-based generator, and the two marginalization variants (RAG-Sequence and RAG-Token).

이 노트북은 작은 사실 코퍼스, dense retriever (임베딩된 패시지에 대한 코사인 유사도), 템플릿 기반 generator, 그리고 두 가지 marginalization 변종 (RAG-Sequence와 RAG-Token)을 갖춘 작은 end-to-end RAG 파이프라인을 구현합니다.

We illustrate: (1) corpus → embeddings, (2) query → top-K retrieval, (3) per-document generator scores, (4) RAG-Sequence vs RAG-Token marginal probabilities, (5) retrieval-similarity heatmap.

구현 흐름: (1) 코퍼스 → 임베딩, (2) 쿼리 → top-K 검색, (3) 문서별 생성기 점수, (4) RAG-Sequence vs RAG-Token marginal 확률, (5) 검색 유사도 히트맵.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
np.random.seed(42)

## Part 1: Tiny Corpus and Bag-of-Words Embeddings / 작은 코퍼스와 BoW 임베딩

We build a 20-fact corpus across history, science, and literature. To stay self-contained (no PyTorch/transformers download), each passage is embedded with **L2-normalized bag-of-words vectors**. Inner product on these vectors is exactly cosine similarity — the same MIPS objective DPR uses, just with hand-crafted features instead of BERT.

20개의 사실로 구성된 코퍼스(역사, 과학, 문학)를 만듭니다. 외부 다운로드 없이 self-contained하게 유지하기 위해 각 패시지를 **L2-정규화된 bag-of-words 벡터**로 임베딩합니다. 이렇게 하면 내적이 정확히 코사인 유사도가 되며, DPR이 사용하는 MIPS 목적과 동일합니다 (BERT 대신 수작업 특징을 사용).

In [ ]:
# Tiny knowledge corpus (20 passages).
corpus = [
    "Marie Curie discovered the elements polonium and radium and won two Nobel Prizes.",
    "Albert Einstein published the special theory of relativity in 1905.",
    "Isaac Newton formulated the laws of motion and universal gravitation in the Principia.",
    "Charles Darwin proposed natural selection in On the Origin of Species in 1859.",
    "Ada Lovelace wrote the first algorithm intended for a machine, Babbage's Analytical Engine.",
    "Alan Turing designed the Turing machine and helped break the Enigma cipher.",
    "The Eiffel Tower was completed in Paris in 1889 for the World's Fair.",
    "The Great Wall of China was built across many dynasties to defend the northern border.",
    "William Shakespeare wrote Hamlet, Macbeth, and Romeo and Juliet.",
    "Jane Austen wrote Pride and Prejudice and Sense and Sensibility.",
    "Ernest Hemingway wrote The Old Man and the Sea and A Farewell to Arms.",
    "Leo Tolstoy wrote War and Peace and Anna Karenina.",
    "Photosynthesis converts carbon dioxide and water into glucose and oxygen using light.",
    "DNA carries genetic information in a double helix structure.",
    "The mitochondrion is the powerhouse of the cell, generating ATP.",
    "Mount Everest is the highest mountain on Earth at 8848 meters.",
    "The Amazon River is the largest river by discharge volume in the world.",
    "The capital of France is Paris and the capital of Japan is Tokyo.",
    "World War II ended in 1945 with the surrender of Germany and Japan.",
    "The first man on the Moon was Neil Armstrong in July 1969.",
]
N_DOCS = len(corpus)
print(f"Corpus size / 코퍼스 크기: {N_DOCS}")

def tokenize(s):
    """Lowercase tokenizer that strips punctuation."""
    return [t.strip(".,!?;:'\"()").lower() for t in s.split() if t.strip(".,!?;:'\"()")]

# Build vocabulary.
vocab = sorted({tok for doc in corpus for tok in tokenize(doc)})
vocab_idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
print(f"Vocabulary size / 어휘 크기: {V}")

def embed(text):
    """L2-normalized bag-of-words vector. Inner product = cosine similarity."""
    v = np.zeros(V)
    for tok in tokenize(text):
        if tok in vocab_idx:
            v[vocab_idx[tok]] += 1.0
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

# Document index (the 'non-parametric memory').
D = np.stack([embed(doc) for doc in corpus])
print(f"Document index shape / 문서 인덱스 모양: {D.shape}")

## Part 2: Dense Retriever — Top-K via MIPS / Dense 리트리버 — MIPS로 top-K

We implement DPR's retrieval score $p_\eta(z|x) \propto \exp(d(z)^\top q(x))$ and select the top-$K$ passages. With L2-normalized vectors, $d(z)^\top q(x)$ is the cosine similarity in $[-1, 1]$. Softmax over the top-$K$ scores produces the truncated retrieval distribution that RAG marginalizes over.

DPR의 검색 점수 $p_\eta(z|x) \propto \exp(d(z)^\top q(x))$를 구현하고 top-$K$를 선택합니다. L2-정규화된 벡터에서 내적은 $[-1, 1]$ 범위의 코사인 유사도입니다. Top-$K$ 점수에 softmax를 적용해 RAG가 marginalize할 truncated 분포를 얻습니다.

In [ ]:
def retrieve(query, K=5, temperature=1.0):
    """Retrieve top-K documents and return (indices, retriever_probs, scores).

    Implements DPR's score p_eta(z|x) proportional to exp(d(z)^T q(x)) followed by
    a top-K truncation, which together approximate the full softmax over 21M passages
    in the original RAG paper.
    """
    q = embed(query)
    scores = D @ q  # shape (N_DOCS,) — MIPS over the index.
    topk = np.argsort(-scores)[:K]
    topk_scores = scores[topk] / temperature
    # Softmax over top-K (truncated marginalization support).
    e = np.exp(topk_scores - topk_scores.max())
    probs = e / e.sum()
    return topk, probs, scores

query = "Who wrote A Farewell to Arms?"
topk_idx, p_z_given_x, full_scores = retrieve(query, K=5, temperature=0.3)

print(f"Query / 쿼리: {query}\n")
print(f"Top-K retrieved (with p_eta(z|x)) / 검색 결과 (확률 포함):")
for i, (idx, p) in enumerate(zip(topk_idx, p_z_given_x)):
    print(f"  [{i+1}] p={p:.3f}  cos={full_scores[idx]:+.3f}  | {corpus[idx]}")

## Part 3: Generator and Marginalization (RAG-Sequence vs RAG-Token) / 생성기와 Marginalization

We replace BART with a **toy generator** that scores candidate answers based on token-level overlap between the candidate and the conditioning passage. Concretely, for a candidate answer $y$ and passage $z$, we compute $p_\theta(y|x,z) = \prod_i p_\theta(y_i | x, z, y_{<i})$ where each token's probability is high when the token appears in $z$ and low otherwise. This is a deliberate caricature of BART's behavior — it captures the qualitative property that **a passage which mentions the answer makes that answer probable** — and it is sufficient to demonstrate the two marginalization formulas.

BART 대신 **장난감 generator**를 사용합니다. 후보 답 $y$와 패시지 $z$에 대해 $p_\theta(y|x,z) = \prod_i p_\theta(y_i | x, z, y_{<i})$를 계산하며, 각 토큰의 확률은 토큰이 $z$에 등장하면 높고 그렇지 않으면 낮도록 설정합니다. 이는 BART 동작의 의도적인 단순화이지만 **답을 언급한 패시지가 그 답을 더 가능성 있게 만든다**는 정성적 성질을 포착하며, 두 marginalization 식을 시연하기에 충분합니다.

Then we compute:
- $p_\text{RAG-Seq}(y|x) = \sum_z p_\eta(z|x) \prod_i p_\theta(y_i|x,z,y_{<i})$
- $p_\text{RAG-Tok}(y|x) = \prod_i \sum_z p_\eta(z|x) p_\theta(y_i|x,z,y_{<i})$

In [ ]:
def token_prob(token, passage, hit=0.85, miss=0.05):
    """Toy per-token generator probability conditioned on a passage.

    Returns a high probability if the token appears in the passage, low otherwise.
    Mimics the qualitative behavior of BART conditioned on a retrieved doc.
    """
    passage_tokens = set(tokenize(passage))
    return hit if token.lower() in passage_tokens else miss

def per_token_probs(answer_tokens, passage):
    """Vector of p_theta(y_i | x, z, y_{<i}) for each token."""
    return np.array([token_prob(t, passage) for t in answer_tokens])

def rag_sequence_prob(answer_tokens, topk_idx, p_z):
    """p_RAG-Seq(y|x) = sum_z p(z|x) * prod_i p_theta(y_i|x,z,y_<i).

    Marginalize OUTSIDE the product. One document conditions the whole sequence.
    Returns (marginal_prob, per_doc_seq_probs).
    """
    per_doc_seq = []
    for idx in topk_idx:
        token_ps = per_token_probs(answer_tokens, corpus[idx])
        per_doc_seq.append(np.prod(token_ps))
    per_doc_seq = np.array(per_doc_seq)
    return float(np.sum(p_z * per_doc_seq)), per_doc_seq

def rag_token_prob(answer_tokens, topk_idx, p_z):
    """p_RAG-Tok(y|x) = prod_i sum_z p(z|x) * p_theta(y_i|x,z,y_<i).

    Marginalize INSIDE the product. Each token can be supported by a different doc.
    Returns (marginal_prob, per_token_marginals).
    """
    per_token = np.zeros(len(answer_tokens))
    for j, t in enumerate(answer_tokens):
        s = 0.0
        for idx, p in zip(topk_idx, p_z):
            s += p * token_prob(t, corpus[idx])
        per_token[j] = s
    return float(np.prod(per_token)), per_token

# Score a few candidate answers for the Hemingway query.
candidates = {
    "Ernest Hemingway": ["Ernest", "Hemingway"],
    "Leo Tolstoy":      ["Leo", "Tolstoy"],
    "Jane Austen":      ["Jane", "Austen"],
    "Marie Curie":      ["Marie", "Curie"],
}

print(f"Query / 쿼리: {query}\n")
print(f"{'Candidate / 후보':<22} {'RAG-Seq p(y|x)':>16} {'RAG-Tok p(y|x)':>16}")
print("-" * 56)
results = {}
for name, toks in candidates.items():
    seq_p, _ = rag_sequence_prob(toks, topk_idx, p_z_given_x)
    tok_p, _ = rag_token_prob(toks, topk_idx, p_z_given_x)
    results[name] = (seq_p, tok_p)
    print(f"{name:<22} {seq_p:>16.4f} {tok_p:>16.4f}")
best = max(results, key=lambda k: results[k][0])
print(f"\nPredicted answer (RAG-Seq argmax) / 예측 답 (RAG-Seq argmax): {best}")

## Part 4: Retrieval Similarity Heatmap / 검색 유사도 히트맵

We visualize cosine similarity between a set of queries and every passage in the index. Bright cells show retrieval focus — exactly the signal RAG's retriever is trained to sharpen via end-task gradients.

여러 쿼리와 인덱스의 모든 패시지 사이의 코사인 유사도를 시각화합니다. 밝은 셀은 검색이 집중된 위치를 나타내며, 이는 RAG retriever가 end-task gradient를 통해 날카롭게 학습하는 신호와 정확히 일치합니다.

In [ ]:
queries = [
    "Who wrote A Farewell to Arms?",
    "What did Marie Curie discover?",
    "Who proposed natural selection?",
    "What is the highest mountain on Earth?",
    "What is the capital of France?",
    "Who was the first man on the Moon?",
]
Q = np.stack([embed(q) for q in queries])
sim = Q @ D.T  # (n_queries, N_DOCS)

fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(sim, aspect='auto', cmap='viridis', vmin=0, vmax=sim.max())
ax.set_xticks(range(N_DOCS))
ax.set_xticklabels([f"d{i}" for i in range(N_DOCS)], fontsize=9)
ax.set_yticks(range(len(queries)))
ax.set_yticklabels(queries, fontsize=9)
ax.set_xlabel("Document index / 문서 인덱스")
ax.set_title("Retriever cosine similarity: queries vs documents / 쿼리-문서 코사인 유사도")
plt.colorbar(im, ax=ax, label="cos(q, d)")
# Annotate top-1 doc for each query.
for i in range(len(queries)):
    j = int(np.argmax(sim[i]))
    ax.text(j, i, "*", ha='center', va='center', color='white', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop-1 retrieved per query / 쿼리별 top-1:")
for q, row in zip(queries, sim):
    j = int(np.argmax(row))
    print(f"  Q: {q}\n     -> d{j} (cos={row[j]:.3f}): {corpus[j]}")

## Part 5: RAG-Sequence vs RAG-Token — Multi-Fact Composition / 다중 사실 결합

Section 4.3 of the paper shows RAG-Token wins on Jeopardy because **two facts from different documents** must be combined. We replicate the qualitative effect: ask which two novels Hemingway wrote, where the answer tokens ("The Old Man and the Sea", "A Farewell to Arms") may be split across passages. We construct a corpus split where Hemingway's two novels are mentioned in **different documents** to make the multi-doc effect visible.

논문의 §4.3은 Jeopardy에서 RAG-Token이 우세함을 보입니다 — **서로 다른 문서의 두 사실**을 결합해야 하기 때문입니다. 그 효과를 정성적으로 재현합니다. Hemingway의 두 소설을 **서로 다른 문서**에 분산해 배치한 뒤 multi-doc 효과를 시각화합니다.

In [ ]:
# Split corpus: each Hemingway novel in a different passage.
split_corpus = list(corpus)
split_corpus[10] = "Ernest Hemingway wrote The Old Man and the Sea, a famous novella."
split_corpus.append("A Farewell to Arms is a 1929 novel by Ernest Hemingway set during World War I.")

vocab2 = sorted({tok for doc in split_corpus for tok in tokenize(doc)})
vocab_idx2 = {w: i for i, w in enumerate(vocab2)}
V2 = len(vocab2)

def embed2(text):
    v = np.zeros(V2)
    for tok in tokenize(text):
        if tok in vocab_idx2:
            v[vocab_idx2[tok]] += 1.0
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

D2 = np.stack([embed2(doc) for doc in split_corpus])

def retrieve2(query, K=5, temperature=0.3):
    q = embed2(query)
    scores = D2 @ q
    topk = np.argsort(-scores)[:K]
    s = scores[topk] / temperature
    e = np.exp(s - s.max())
    return topk, e / e.sum()

q2 = "What two novels did Hemingway write?"
topk2, p_z2 = retrieve2(q2, K=5)
print(f"Query / 쿼리: {q2}")
for idx, p in zip(topk2, p_z2):
    print(f"  p={p:.3f} | {split_corpus[idx]}")

# Candidate answer that requires combining two passages.
answer = ["The", "Old", "Man", "and", "the", "Sea", "and", "A", "Farewell", "to", "Arms"]

def token_prob2(token, passage, hit=0.85, miss=0.05):
    return hit if token.lower() in set(tokenize(passage)) else miss

# RAG-Sequence: pick one doc, score the whole sequence, mix.
per_doc = np.array([
    np.prod([token_prob2(t, split_corpus[idx]) for t in answer]) for idx in topk2
])
rag_seq = float((p_z2 * per_doc).sum())

# RAG-Token: at each token, marginalize over docs.
per_tok = np.zeros(len(answer))
per_tok_doc_post = np.zeros((len(answer), len(topk2)))
for j, t in enumerate(answer):
    contributions = np.array([p_z2[k] * token_prob2(t, split_corpus[topk2[k]]) for k in range(len(topk2))])
    per_tok[j] = contributions.sum()
    per_tok_doc_post[j] = contributions / per_tok[j] if per_tok[j] > 0 else 0
rag_tok = float(np.prod(per_tok))

print(f"\nRAG-Sequence p(y|x) = {rag_seq:.3e}")
print(f"RAG-Token    p(y|x) = {rag_tok:.3e}")
print(f"Ratio (Tok / Seq) / 비율: {rag_tok / rag_seq:.2f}x  -- RAG-Token wins on multi-fact composition / 다중 사실 결합에서 우세")

# Per-token document posterior heatmap (mirrors paper Figure 2).
fig, ax = plt.subplots(figsize=(11, 4.5))
im = ax.imshow(per_tok_doc_post.T, aspect='auto', cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(answer)))
ax.set_xticklabels(answer, rotation=30, ha='right')
ax.set_yticks(range(len(topk2)))
ax.set_yticklabels([f"d{idx}: {split_corpus[idx][:50]}..." for idx in topk2], fontsize=8)
ax.set_xlabel("Generated token / 생성 토큰")
ax.set_title("RAG-Token: per-token document posterior p(z | x, y_i, y_<i)\n(mirrors paper Figure 2 — Hemingway example)")
plt.colorbar(im, ax=ax, label='posterior')
plt.tight_layout()
plt.show()

## Part 6: Effect of Top-K and Index Hot-Swapping / Top-K 효과와 인덱스 핫스왑

We reproduce two qualitative findings from the paper:

1. **More retrieved docs help up to a point** (Figure 3): we sweep $K$ and plot the marginal probability of the gold answer.
2. **Index hot-swapping** (§4.5): we replace one document in the index with a contradicting newer fact and show the predicted answer changes — without retraining.

논문의 두 가지 정성적 발견을 재현합니다.

1. **검색 문서 수 증가 효과** (그림 3): $K$를 변화시키며 정답의 marginal 확률을 그립니다.
2. **인덱스 핫스왑** (§4.5): 인덱스의 한 문서를 반대 사실로 교체해 재학습 없이 예측이 바뀜을 보입니다.

In [ ]:
# (1) Effect of K on RAG-Sequence marginal for the gold answer.
gold_answer = ["Ernest", "Hemingway"]
Ks = list(range(1, 11))
rag_seq_curve = []
rag_tok_curve = []
for K in Ks:
    topk_K, p_K, _ = retrieve(query, K=K, temperature=0.3)
    s, _ = rag_sequence_prob(gold_answer, topk_K, p_K)
    t, _ = rag_token_prob(gold_answer, topk_K, p_K)
    rag_seq_curve.append(s)
    rag_tok_curve.append(t)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(Ks, rag_seq_curve, 'o-', label='RAG-Sequence')
axes[0].plot(Ks, rag_tok_curve, 's-', label='RAG-Token')
axes[0].set_xlabel('K (retrieved docs) / 검색 문서 수')
axes[0].set_ylabel('p(gold | x)')
axes[0].set_title('Effect of K on marginal probability\nK 변화에 따른 marginal 확률')
axes[0].legend()
axes[0].grid(alpha=0.3)

# (2) Index hot-swap: pretend we update the corpus to assert Tolstoy wrote A Farewell to Arms (counterfactual).
swap_corpus = list(corpus)
swap_corpus[10] = "Leo Tolstoy wrote A Farewell to Arms."  # COUNTERFACTUAL replacement
vocab_s = sorted({tok for doc in swap_corpus for tok in tokenize(doc)})
vocab_idx_s = {w: i for i, w in enumerate(vocab_s)}
Vs = len(vocab_s)
def embed_s(text):
    v = np.zeros(Vs)
    for tok in tokenize(text):
        if tok in vocab_idx_s:
            v[vocab_idx_s[tok]] += 1.0
    n = np.linalg.norm(v)
    return v / n if n > 0 else v
Ds = np.stack([embed_s(doc) for doc in swap_corpus])

def score_with_index(corpus_arr, D_arr, embed_fn, query, candidates, K=5):
    q = embed_fn(query)
    scores = D_arr @ q
    topk = np.argsort(-scores)[:K]
    s = scores[topk] / 0.3
    e = np.exp(s - s.max()); p = e / e.sum()
    out = {}
    for name, toks in candidates.items():
        per_doc = np.array([np.prod([token_prob(t, corpus_arr[idx]) for t in toks]) for idx in topk])
        out[name] = float((p * per_doc).sum())
    return out

orig_scores = score_with_index(corpus, D, embed, query, candidates, K=5)
swap_scores = score_with_index(swap_corpus, Ds, embed_s, query, candidates, K=5)

names = list(candidates.keys())
x = np.arange(len(names))
w = 0.35
axes[1].bar(x - w/2, [orig_scores[n] for n in names], w, label='Original index / 원본 인덱스')
axes[1].bar(x + w/2, [swap_scores[n] for n in names], w, label='Swapped index / 교체된 인덱스')
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=20, ha='right')
axes[1].set_ylabel('RAG-Seq p(y | x)')
axes[1].set_title('Index hot-swap changes the answer\n인덱스 교체로 답이 바뀜')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("Original argmax / 원본 argmax:", max(orig_scores, key=orig_scores.get))
print("Swapped  argmax / 교체 argmax: ", max(swap_scores, key=swap_scores.get))
print("\n=> RAG's knowledge can be updated by editing the document index alone, with no model retraining.")
print("   RAG의 지식은 문서 인덱스 편집만으로 갱신 가능 — 모델 재학습 불필요.")

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | This Notebook / 이 노트북 |
|---|---|---|
| Document encoder | BERT-base bi-encoder (DPR) | L2-normalized bag-of-words |
| Query encoder | BERT-base (frozen doc, fine-tuned query) | Same BoW embedding |
| Index | FAISS HNSW over 21M Wikipedia passages | NumPy matmul over 20 facts |
| Generator | BART-large (400M params) | Token-overlap toy generator |
| Retrieval score | $p_\eta(z\|x) \propto \exp(d(z)^\top q(x))$ | Same form (cosine + softmax over top-K) |
| RAG-Sequence | $\sum_z p(z\|x) \prod_i p(y_i\|x,z,y_{<i})$ | Implemented in `rag_sequence_prob` |
| RAG-Token | $\prod_i \sum_z p(z\|x) p(y_i\|x,z,y_{<i})$ | Implemented in `rag_token_prob` |
| Index hot-swap | Wikipedia 2016 vs 2018 dumps | Counterfactual passage replacement |
| Marginalization scale | top-K with K∈{5,10} | top-K sweep over K∈[1,10] |

**Key takeaway / 핵심 시사점**: RAG's three pieces — dense retriever, generator, and marginalization — are conceptually independent and compose cleanly. Even with toy components (BoW + token-overlap), the qualitative behaviors of the paper survive: top-K retrieval focuses on the right passage, RAG-Token outperforms RAG-Sequence on multi-fact composition, and index hot-swapping flips the predicted answer without retraining.

RAG의 세 컴포넌트(dense retriever, generator, marginalization)는 개념적으로 독립적이며 깔끔하게 조합됩니다. 장난감 컴포넌트(BoW + 토큰 중첩)에서도 논문의 정성적 동작이 살아남습니다: top-K 검색이 올바른 패시지에 집중하고, RAG-Token이 다중 사실 결합에서 우세하며, 인덱스 핫스왑만으로 예측이 바뀝니다.